# Phase 8 — 로컬 파일럿과 운영 검증

기존 원문·인덱스와 Phase 6 대화를 사용합니다. **기본 Run All은 API 0회**입니다. 개발 세트 성적과 새 파일럿 사람 검토를 구분합니다.

최초 업데이트 후에는 기존 Python 커널을 Restart하세요. 원문·인덱스 재구축과 추가 UI 패키지 설치는 필요하지 않습니다.

In [ ]:
from pathlib import Path
import json, sys
from IPython.display import display, Markdown

PROJECT_ROOT = None
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd()/'rfp_phase5_local_20261111']
ROOT = (Path(PROJECT_ROOT).expanduser().resolve() if PROJECT_ROOT else
        next((p.resolve() for p in candidates if (p/'rfp_phase8/pilot.py').is_file()), None))
if ROOT is None:
    raise FileNotFoundError('PROJECT_ROOT에 rfp_phase5_local_20261111 폴더를 지정하세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rfp_phase5.runtime import bootstrap_phase5
from rfp_phase8.pilot import PilotSession, readiness, session_report
if 'env' in globals():
    env['index'].close()
env = bootstrap_phase5(ROOT)
if 'pilot' not in globals() or pilot.root != ROOT:
    pilot = PilotSession(env)
else:
    pilot.attach(env)
print('Python:', sys.executable)
print('프로젝트:', ROOT)
print('이번 커널 세션:', pilot.session_id)
display(readiness(env))


## 1. 원문 검색 동작 확인

첫 대표 문서에 일반적인 예산·기간 질문을 검색합니다. API 생성·개발 정답 파일 조회 없이 동작만 확인합니다. 같은 커널에서 다시 Run All 해도 기존 호출 누적값은 유지됩니다.

In [ ]:
pilot.configure_generation(enabled=False)
probe_id = 'notebook-offline-probe-v1'
if probe_id not in pilot._requests:
    pilot.select([pilot.chat.list_documents()[0]['doc_id']], filters={})
probe = pilot.submit('사업 예산과 과업기간을 알려줘.', request_id=probe_id)
display({'상태': probe['result']['status'], '실행': probe['result'].get('execution_status'),
         '이번 호출': probe['usage_delta'], '생성 답변 정답 판정': None})
for chunk in probe['result'].get('source_chunks', [])[:2]:
    display({'문서': chunk.get('file_name'), '위치': chunk.get('locator'), '원문 후보': chunk['text'][:700]})


## 2. 회귀 테스트

합성 RFP와 대역 응답만 사용합니다. 호출 누적, 비교 사전 한도 검사, 새로고침·중복 제출, 실패 중단, 검토 연결을 확인합니다.

In [ ]:
import subprocess
checks = subprocess.run([sys.executable, '-m', 'unittest', 'rfp_phase8.test_pilot', '-q'],
                        cwd=ROOT, capture_output=True, text=True)
print(checks.stdout + checks.stderr)
checks.check_returncode()


## 3. 실제 파일럿 — 기본 꺼짐

아래 문서 ID와 질문은 직접 선택합니다. 개발 정답·기대 문서 목록으로 실행 범위를 만들지 않습니다. `False` Run All 후 `True` Run All 또는 실제 질문 셀만 실행할 수 있습니다. 같은 배치 ID·질문 순번은 재실행해도 중복 호출하지 않습니다. **새 질문·문서 조합은 PILOT_BATCH_ID도 변경**하세요. 오류가 있으면 진단 확인 후 `pilot.resume_after_error()`를 명시적으로 실행하세요.

In [ ]:
RUN_LIVE_PILOT = False
PILOT_BATCH_ID = 'pilot-01'
PILOT_DOC_IDS = []  # 아래 목록에서 실제 사용할 문서 ID만 복사
PILOT_QUESTIONS = []  # 예: ['예산의 부가세 조건은?', '그 기간에 안정화도 포함돼?']

catalog = pilot.chat.list_documents()
display([{'doc_id': d['doc_id'], '사업명': d['project_title'], '기관': d.get('agency'),
          '연도': d.get('document_year')} for d in catalog[:15]])
print('전체 목록은 pilot.chat.list_documents(검색어)로 조회합니다.')


In [ ]:
live_records = []
if RUN_LIVE_PILOT:
    if not PILOT_DOC_IDS or not PILOT_QUESTIONS:
        raise ValueError('실제 사용할 문서 ID와 질문을 먼저 입력하세요.')
    from rfp_phase5_recovery.runner import load_api_key
    env['registry'].resolve_doc_ids(PILOT_DOC_IDS)
    load_api_key(ROOT)
    pilot.configure_generation(enabled=True, approved_doc_ids=PILOT_DOC_IDS)
    pilot.select(PILOT_DOC_IDS, filters={})
    print('API 전송 허용 문서:', pilot.approved_doc_ids)
    for i, question in enumerate(PILOT_QUESTIONS):
        record = pilot.submit(question, request_id=f'{PILOT_BATCH_ID}:live:{i}')
        live_records.append(record)
        display(Markdown(record['result'].get('markdown', '')))
        display({'턴': record['turn_id'], '상태': record['result']['status'],
                 '실행': record['result'].get('execution_status'), '사용량': record['usage_delta'],
                 '진단': record['result'].get('error_diagnostic'),
                 '검토 파일': str(pilot.folder/'turns'/record['turn_id']/'human_review.json')})
        if pilot.halted:
            print('실행을 중단했습니다. 원인을 확인한 뒤 명시적으로 계속하세요.')
            break
else:
    pilot.configure_generation(enabled=False)
    print('실제 파일럿 생성 꺼짐. 추가 API 호출 0회.')


## 4. 실행과 사람 검토 집계

출력된 각 턴의 `human_review.json`에 검토자와 다섯 개의 true/false 판정을 기록한 뒤 이 셀을 다시 실행하세요. 미판정은 null입니다. 인용 유효성 검사와 의미적 정답은 별도이며, 신규 질문·문서 여부는 아직 미검증입니다. 기본 단가가 null이므로 비용을 0달러로 표시하지 않습니다.

In [ ]:
report = session_report(pilot.folder)
print('현재 실행 기록:', pilot.folder)
display(report)
print('세션 누적 사용량:', pilot.chat.llm.usage_report())
print('새 문서·질문의 독립 품질 검증:', report['independent_quality_verified'])


## 5. 로컬 화면에서 사용

새 터미널에서 기존 가상환경을 활성화한 뒤 아래 명령을 실행하고 표시된 127.0.0.1 주소를 여세요. 화면은 별도의 서버 세션과 상한을 사용합니다. 새로고침은 질문을 재실행하지 않습니다. 종료는 Ctrl+C입니다.

```bash
python rfp_phase5_local_20261111/scripts/phase8_pilot.py --serve
```

Phase 7의 남은 11/141 원문 구간과 저장 요약이 없는 개발 사례 9건은 알려진 한계로 유지합니다. Phase 8 실제 사용자 검토·독립 평가를 완료한 것으로 자동 판정하지 않습니다.

In [ ]:
# 조회를 마친 읽기 전용 인덱스만 닫습니다. 다음 Run All에서 새 인덱스를 연결합니다.
env['index'].close()
